Assessment 2 — Experimental Report
=========================================
A simulated audit study of the intersection of name-ethnicity (Turkish-origin vs. English-origin) and gender on CV callback rates in UK corporate hiring.

LIS MASc | 2025/26 | FT Trials & Errors | 25000148737

Overview
--------
This script generates simulated callback data for a 2 (name-ethnicity: Turkish, English) x 2 (gender: Female, Male) between-CV factorial audit study, and then analyses the data with:

    1. Descriptive statistics per cell, with 95% Wilson confidence intervals (Newcombe, 1998).
    2. A Pearson chi-square test of the 2 x 2 ethnicity x callback contingency table.
    3. A pre-registered binary logistic regression of callback on name-ethnicity, gender, and their interaction, reporting odds ratios with 95% Wald confidence intervals.
    4. A supplementary Bayesian analysis using uniform Beta(1, 1) priors on each cell's callback probability, yielding posterior means and 95% credible intervals, and the posterior probability that Turkish-named CVs are less likely to be called back than English-named CVs.

Simulation parameters are fixed a priori from effect sizes reported in Kaas and Manger (2012) and Pedulla (2014). The random seed is fixed at 42 to guarantee exact reproducibility.

Outputs 
---------------------------------------
    simulated_data.csv         - the 800 simulated CVs
    figure_callback_rates.png  - Figure 1 of the report
    regression_summary.txt     - full logistic regression 
    analysis_log.txt           - all numbers reported 



In [1]:
from __future__ import annotations

import io
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

# 1. REPRODUCIBILITY
SEED = 42
rng = np.random.default_rng(SEED)

# 2. SIMULATION PARAMETERS 
N_PER_CELL = 200               # 200 x 4 cells = N = 800 CVs
BASELINE_P = 0.25              # baseline callback probability (English male)
OR_TURKISH = 0.55              # main effect of Turkish-origin name
OR_FEMALE = 0.95               # small main effect of female-signalled name
OR_INTERACTION = 0.75          # additional Turkish x Female disadvantage

# 3. BUILD THE SIMULATED DATASET
rows = []
for ethnicity in ("English", "Turkish"):
    for gender in ("Male", "Female"):
        rows.extend({"ethnicity": ethnicity, "gender": gender}
                    for _ in range(N_PER_CELL))
df = pd.DataFrame(rows)

# Dummy-code the two predictors (reference: English, Male).
df["turkish"] = (df["ethnicity"] == "Turkish").astype(int)
df["female"] = (df["gender"] == "Female").astype(int)

# Compute each CV's probability of a callback using the log-odds equation
#   logit(p) = b0 + b1*turkish + b2*female + b3*(turkish x female).
baseline_logodds = np.log(BASELINE_P / (1 - BASELINE_P))
log_odds = (
    baseline_logodds
    + np.log(OR_TURKISH) * df["turkish"]
    + np.log(OR_FEMALE) * df["female"]
    + np.log(OR_INTERACTION) * df["turkish"] * df["female"]
)
p_callback = 1 / (1 + np.exp(-log_odds))
df["callback"] = rng.binomial(1, p_callback)

# Persist the raw dataset for the supplementary materials.
df.to_csv("simulated_data.csv", index=False)

# 4. HELPERS
def wilson_ci(successes: int, n: int, z: float = 1.96) -> tuple[float, float]:
    """Wilson score interval for a binomial proportion (Newcombe, 1998).

    More accurate than the normal-approximation interval for small n or
    proportions near 0 or 1.
    """
    if n == 0:
        return (0.0, 0.0)
    p = successes / n
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2))) / denom
    return (centre - half, centre + half)

# 5. DESCRIPTIVE STATISTICS
descriptives = (
    df.groupby(["ethnicity", "gender"], as_index=False)["callback"]
      .agg(callback_rate="mean", callbacks="sum", n="count")
)
ci = descriptives.apply(
    lambda r: wilson_ci(int(r["callbacks"]), int(r["n"])), axis=1
)
descriptives["ci_low"] = [c[0] for c in ci]
descriptives["ci_high"] = [c[1] for c in ci]

# 6. PEARSON CHI-SQUARE TEST
contingency = pd.crosstab(df["ethnicity"], df["callback"])
chi2, p_chi, dof, _ = stats.chi2_contingency(contingency, correction=False)

# 7. PRE-REGISTERED LOGISTIC REGRESSION (MAIN INFERENTIAL TEST)
model = smf.logit("callback ~ turkish * female", data=df).fit(disp=False)

conf = model.conf_int()
conf.columns = ["CI_low", "CI_high"]
odds_ratios = pd.DataFrame(
    {
        "OR": np.exp(model.params),
        "CI_low": np.exp(conf["CI_low"]),
        "CI_high": np.exp(conf["CI_high"]),
        "p_value": model.pvalues,
    }
).round(3)

# 8. SUPPLEMENTARY BAYESIAN ANALYSIS (Beta-Binomial posteriors)
# Uniform Beta(1, 1) priors on each cell's callback probability;
# posterior is Beta(1 + callbacks, 1 + (n - callbacks)).
N_DRAWS = 50_000
posteriors = {}
for _, r in descriptives.iterrows():
    alpha = 1 + r["callbacks"]
    beta = 1 + (r["n"] - r["callbacks"])
    posteriors[(r["ethnicity"], r["gender"])] = rng.beta(alpha, beta, N_DRAWS)

english_rate = np.mean(
    [posteriors[("English", "Male")], posteriors[("English", "Female")]], axis=0
)
turkish_rate = np.mean(
    [posteriors[("Turkish", "Male")], posteriors[("Turkish", "Female")]], axis=0
)
post_prob_turkish_lower = float(np.mean(turkish_rate < english_rate))

bayes_summary = pd.DataFrame(
    [
        {
            "cell": f"{eth}-{gen}",
            "posterior_mean": posteriors[(eth, gen)].mean(),
            "CI_2.5": np.quantile(posteriors[(eth, gen)], 0.025),
            "CI_97.5": np.quantile(posteriors[(eth, gen)], 0.975),
        }
        for eth, gen in posteriors
    ]
)

# 9. FIGURE: CALLBACK RATES BY CONDITION, 95% WILSON CIs
sns.set_theme(style="whitegrid", context="paper")
fig, ax = plt.subplots(figsize=(7, 4.5))

ethnicities = ["English", "Turkish"]
genders = ["Male", "Female"]
x = np.arange(len(ethnicities))
width = 0.35
colours = {"Male": "#4C78A8", "Female": "#F58518"}

for i, gender in enumerate(genders):
    sub = (descriptives[descriptives["gender"] == gender]
           .set_index("ethnicity")
           .loc[ethnicities])
    ax.bar(
        x + (i - 0.5) * width,
        sub["callback_rate"].values,
        width,
        label=gender,
        color=colours[gender],
        yerr=[
            sub["callback_rate"].values - sub["ci_low"].values,
            sub["ci_high"].values - sub["callback_rate"].values,
        ],
        capsize=4,
        edgecolor="black",
        linewidth=0.6,
    )

ax.set_xticks(x)
ax.set_xticklabels(ethnicities)
ax.set_ylabel("Callback rate")
ax.set_xlabel("Name-ethnicity condition")
ax.set_title(
    "Figure 1. Simulated callback rates by name-ethnicity and gender.\n"
    "Error bars: 95% Wilson confidence intervals."
)
ax.set_ylim(0, 0.36)
ax.legend(title="Gender", frameon=True)
plt.tight_layout()
plt.savefig("figure_callback_rates.png", dpi=250, bbox_inches="tight")
plt.close(fig)

# 10. WRITE OUTPUTS
with open("regression_summary.txt", "w", encoding="utf-8") as f:
    f.write(str(model.summary()))

with io.StringIO() as buf, redirect_stdout(buf):
    print("=" * 72)
    print("AUDIT STUDY — ANALYSIS LOG (seed = {0})".format(SEED))
    print("=" * 72)
    print()
    print("Descriptive statistics (callback rate by condition):")
    print(descriptives.round(3).to_string(index=False))
    print()
    print(f"Pearson chi-square (ethnicity x callback): "
          f"chi2({dof}) = {chi2:.2f}, p = {p_chi:.4f}")
    print()
    print("Logistic regression — odds ratios, 95% Wald CIs, p-values:")
    print(odds_ratios.to_string())
    print()
    print(f"Omnibus LLR p-value: {model.llr_pvalue:.4g}")
    print(f"McFadden pseudo R-squared: {model.prsquared:.3f}")
    print()
    print("Bayesian supplementary analysis (Beta-Binomial, 50,000 draws):")
    print(bayes_summary.round(3).to_string(index=False))
    print()
    print(f"Posterior probability that Turkish rate < English rate: "
          f"{post_prob_turkish_lower:.4f}")
    analysis_log = buf.getvalue()

with open("analysis_log.txt", "w", encoding="utf-8") as f:
    f.write(analysis_log)

# Also print to console for immediate inspection.
print(analysis_log)
print("Wrote: simulated_data.csv, figure_callback_rates.png, "
      "regression_summary.txt, analysis_log.txt")

AUDIT STUDY — ANALYSIS LOG (seed = 42)

Descriptive statistics (callback rate by condition):
ethnicity gender  callback_rate  callbacks   n  ci_low  ci_high
  English Female          0.260         52 200   0.204    0.325
  English   Male          0.245         49 200   0.191    0.309
  Turkish Female          0.115         23 200   0.078    0.167
  Turkish   Male          0.140         28 200   0.099    0.195

Pearson chi-square (ethnicity x callback): chi2(1) = 20.31, p = 0.0000

Logistic regression — odds ratios, 95% Wald CIs, p-values:
                   OR  CI_low  CI_high  p_value
Intercept       0.325   0.235    0.448    0.000
turkish         0.502   0.300    0.838    0.008
female          1.083   0.689    1.700    0.730
turkish:female  0.737   0.351    1.550    0.421

Omnibus LLR p-value: 9.115e-05
McFadden pseudo R-squared: 0.027

Bayesian supplementary analysis (Beta-Binomial, 50,000 draws):
          cell  posterior_mean  CI_2.5  CI_97.5
English-Female           0.262   0.204